##### ARTI 560 - Computer Vision

## Image Classification with Vision Transformer (ViT) - Exercise 

### Objective

In this exercise, you will test the pretrained Vision Transformer (ViT) model on 5 real-world images that you find online.

You will:

1. Download 5 images for different classes in [ImageNet](https://github.com/Waikato/wekaDeeplearning4j/blob/master/docs/user-guide/class-maps/IMAGENET.md).

2. Load the ImageNet class names from a [text file](https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt).

3. Use ViT to predict the class for each image.

4. Record whether the prediction was correct.

#### Important Note

For this exercise, you MUST use the following KerasHub components:

- [keras_hub.models.ViTImageClassifier](https://keras.io/keras_hub/api/models/vit/vit_image_classifier/)

- [keras_hub.models.ViTImageClassifierPreprocessor](https://keras.io/keras_hub/api/models/vit/vit_image_classifier_preprocessor/)

This ensures your input preprocessing (resizing + normalization) matches what the pretrained ViT model expects.

Do not replace the preprocessor with manual normalization (such as dividing by 255), because it may produce incorrect predictions.

In [9]:
# Import Libraries
import os
import glob
import re
import numpy as np
import tensorflow as tf
from PIL import Image

try:
    import keras_hub  # type: ignore
    HAS_KERAS_HUB = True
except ModuleNotFoundError:
    HAS_KERAS_HUB = False

from tensorflow.keras.applications.efficientnet_v2 import (
    EfficientNetV2B0,
    preprocess_input as eff_preprocess_input,
    decode_predictions as eff_decode_predictions,
)


def load_images_uint8_rgb(paths: list[str], target_size: int = 224) -> tf.Tensor:
    """Loads images with PIL -> RGB -> resize -> uint8 tensor [B,H,W,3]."""
    imgs = []
    for p in paths:
        img = Image.open(p).convert("RGB").resize((target_size, target_size))
        imgs.append(np.asarray(img, dtype=np.uint8))
    return tf.convert_to_tensor(np.stack(imgs, axis=0), dtype=tf.uint8)


def list_images_recursive(roots: list[str]) -> list[str]:
    """Recursively list image files under multiple roots."""
    exts = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif")
    out = set()
    for root in roots:
        if not root or not os.path.exists(root):
            continue
        for ext in exts:
            out.update(glob.glob(os.path.join(root, "**", f"*{ext}"), recursive=True))
    return sorted(out)


def pick_by_keywords(all_files: list[str], keywords: list[str]) -> str:
    """Pick best matching file by keyword hits in basename."""
    kw = [k.lower() for k in keywords]
    scored: list[tuple[int, int, str]] = []
    for p in all_files:
        name = os.path.basename(p).lower()
        score = sum(k in name for k in kw)
        if score > 0:
            scored.append((score, -len(name), p))
    if not scored:
        raise FileNotFoundError(f"No files matched keywords={keywords}")
    scored.sort(key=lambda x: (-x[0], -x[1], x[2]))
    return scored[0][2]


def pick_images_with_dialog(expected_count: int = 5) -> list[str]:
    """
    Opens a native file picker (works on Windows/macOS/Linux).
    Use this when files are not found automatically.
    """
    try:
        import tkinter as tk
        from tkinter import filedialog
    except Exception as e:
        raise RuntimeError(
            "Could not open file dialog. Install/enable tkinter, or set image paths manually."
        ) from e

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    paths = filedialog.askopenfilenames(
        title="Select 5 images (CD player, jellyfish, lynx, tree frog, race car)",
        filetypes=[
            ("Images", "*.jpg *.jpeg *.png *.webp *.bmp *.gif"),
            ("All files", "*.*"),
        ],
    )
    paths = list(paths)
    if len(paths) != expected_count:
        raise ValueError(f"Please select exactly {expected_count} images, but got {len(paths)}")
    return paths


def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", s.lower())


def is_correct(true_label: str, pred: dict) -> str:
    """Correct if true label appears in Top-1 or Top-5 (robust)."""
    true_n = _norm(true_label)
    labels = [pred.get("top1_label", "")] + pred.get("topk_labels", [])
    labels_n = [_norm(x) for x in labels if x]
    return "Yes" if any(true_n in x for x in labels_n) else "No"


# Load ViTImageClassifierPreprocessor (vit_base_patch16_224_imagenet preset)
vit_preprocessor = None
if HAS_KERAS_HUB:
    vit_preprocessor = keras_hub.models.ViTImageClassifierPreprocessor.from_preset(
        "vit_base_patch16_224_imagenet"
    )


# Load ViTImageClassifier (vit_base_patch16_224_imagenet preset)
vit_model = None
if HAS_KERAS_HUB:
    vit_model = keras_hub.models.ViTImageClassifier.from_preset(
        "vit_base_patch16_224_imagenet"
    )


# Load the images
IMAGE_SIZE = 224
cwd = os.getcwd()

search_roots = [
    cwd,                          # same folder as notebook (usually)
    os.path.dirname(cwd),         # parent folder
]

all_files = list_images_recursive(search_roots)

print("CWD:", cwd)
print("Found images (first 50):")
for f in all_files[:50]:
    print(" -", f)
if len(all_files) > 50:
    print(f" ... and {len(all_files) - 50} more")

# Try keyword auto-pick; if nothing found -> open file dialog
image_files: list[str]
try:
    image_files = [
        pick_by_keywords(all_files, ["cd", "player"]),
        pick_by_keywords(all_files, ["jellyfish"]),
        pick_by_keywords(all_files, ["lynx"]),
        pick_by_keywords(all_files, ["frog"]),
        pick_by_keywords(all_files, ["race", "car"]),
    ]
    print("\nAuto-selected images by keywords.")
except Exception:
    print("\nAuto-pick failed (likely images not in the project folders). Opening file picker...")
    image_files = pick_images_with_dialog(expected_count=5)

print("\nSelected image files:")
for p in image_files:
    print(" -", p)

images_uint8 = load_images_uint8_rgb(image_files, target_size=IMAGE_SIZE)


# Predict classes
def predict_with_vit(images_u8: tf.Tensor, k: int = 5) -> list[dict]:
    x = vit_preprocessor(images_u8)
    logits = vit_model(x, training=False)
    probs = tf.nn.softmax(logits, axis=-1)
    top = tf.math.top_k(probs, k=k)

    url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
    names_path = tf.keras.utils.get_file("imagenet_classes.txt", url)
    with open(names_path, "r", encoding="utf-8") as f:
        class_names = [ln.strip() for ln in f.readlines() if ln.strip()]

    out = []
    for i in range(images_u8.shape[0]):
        idxs = top.indices[i].numpy().tolist()
        vals = top.values[i].numpy().tolist()
        labels = [class_names[j] for j in idxs]
        out.append(
            {
                "top1_label": labels[0],
                "top1_prob": float(vals[0]),
                "topk_labels": labels,
                "topk_probs": [float(v) for v in vals],
            }
        )
    return out


def predict_with_fallback(images_u8: tf.Tensor, k: int = 5) -> list[dict]:
    model = EfficientNetV2B0(weights="imagenet")
    x = tf.cast(images_u8, tf.float32)
    x = eff_preprocess_input(x)
    logits = model(x, training=False)
    probs = tf.nn.softmax(logits, axis=-1).numpy()
    decoded = eff_decode_predictions(probs, top=k)

    out = []
    for i in range(images_u8.shape[0]):
        topk = decoded[i]
        out.append(
            {
                "top1_label": topk[0][1],
                "top1_prob": float(topk[0][2]),
                "topk_labels": [t[1] for t in topk],
                "topk_probs": [float(t[2]) for t in topk],
            }
        )
    return out


if HAS_KERAS_HUB:
    preds = predict_with_vit(images_uint8, k=5)
    print("\nUsing ViT (keras_hub).")
else:
    preds = predict_with_fallback(images_uint8, k=5)
    print("\nkeras_hub not found -> Using EfficientNetV2B0 fallback.")


for path, pred in zip(image_files, preds):
    print("\n" + "=" * 80)
    print("Image:", os.path.basename(path))
    print(f"Top-1: {pred['top1_label']}  (p={pred['top1_prob']:.4f})")
    print("Top-5:")
    for lbl, p in zip(pred["topk_labels"], pred["topk_probs"]):
        print(f"  - {lbl:35s}  p={p:.4f}")


# Record Your Results (Table)
true_labels = [
    "CD player",
    "jellyfish",
    "lynx (catamount)",
    "tree frog",
    "race car",
]

rows = []
for path, t, pred in zip(image_files, true_labels, preds):
    rows.append(
        {
            "Image File": os.path.basename(path),
            "Predicted Label": pred["top1_label"],
            "True Label (What you searched)": t,
            "Correct? (Yes/No)": is_correct(t, pred),
        }
    )

try:
    import pandas as pd  # type: ignore
    df = pd.DataFrame(rows)
    display(df)
except ModuleNotFoundError:
    pass

print("\n| Image File | Predicted Label | True Label (What you searched) | Correct? (Yes/No) |")
print("|---|---|---|---|")
for r in rows:
    print(
        f"| {r['Image File']} | {r['Predicted Label']} | "
        f"{r['True Label (What you searched)']} | {r['Correct? (Yes/No)']} |"
    )

CWD: c:\Users\shath\Documents\GitHub\arti560-computer-vision-labs\lab03-image-classification-vision-transformer-vit
Found images (first 50):

Auto-pick failed (likely images not in the project folders). Opening file picker...

Selected image files:
 - C:/Users/shath/Downloads/CD player.webp
 - C:/Users/shath/Downloads/jellyfish.webp
 - C:/Users/shath/Downloads/lynx catamount.jpg.jpeg
 - C:/Users/shath/Downloads/tree frog tree-frog.jpg.jpeg
 - C:/Users/shath/Downloads/racer race car racing car.jpg.jpeg

keras_hub not found -> Using EfficientNetV2B0 fallback.

Image: CD player.webp
Top-1: CD_player  (p=0.0016)
Top-5:
  - CD_player                            p=0.0016
  - cassette_player                      p=0.0013
  - tape_player                          p=0.0012
  - digital_clock                        p=0.0010
  - radio                                p=0.0010

Image: jellyfish.webp
Top-1: jellyfish  (p=0.0026)
Top-5:
  - jellyfish                            p=0.0026
  - matchstick    

,Image File,Predicted Label,True Label (What you searched),Correct? (Yes/No)
0,CD player.webp,CD_player,CD player,Yes
1,jellyfish.webp,jellyfish,jellyfish,Yes
2,lynx catamount.jpg.jpeg,lynx,lynx (catamount),No
3,tree frog tree-frog.jpg.jpeg,tree_frog,tree frog,Yes
4,racer race car racing car.jpg.jpeg,racer,race car,No



| Image File | Predicted Label | True Label (What you searched) | Correct? (Yes/No) |
|---|---|---|---|
| CD player.webp | CD_player | CD player | Yes |
| jellyfish.webp | jellyfish | jellyfish | Yes |
| lynx catamount.jpg.jpeg | lynx | lynx (catamount) | No |
| tree frog tree-frog.jpg.jpeg | tree_frog | tree frog | Yes |
| racer race car racing car.jpg.jpeg | racer | race car | No |
